# Phase 4: Advanced Forecasting and Model Evaluation

Nested rolling-origin evaluation using the existing availability, feature and baseline pipeline. All outputs and local MLflow experiments remain Git-ignored. No BOM costing or pricing optimization is performed. The experiment remains assumption-based and cannot remove historical revision leakage from the November data snapshot.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").exists()
)
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "scripts"))

from run_phase4 import main  # noqa: E402

results = main()

out = results["output"]

## Nested evaluation protocol

Source-supported lags and the provisional FX rule are inherited from Phase 3. Each inner training set has its own label-release cutoff. Preprocessing fits only inside that training set. The 24 economic predictors and two-option grids are fixed. Outer selection origins are quarterly during 2021–2022; families freeze December 31, 2023. Separate confirmation origins begin January 2024. Failed windows are excluded jointly across all 13 candidates, with complete coverage and tuning ledgers.

In [ ]:
print((out / "experiment_config.json").read_text())
display(
    pd.read_csv(out / "coverage.csv").groupby(["model", "success", "paired"]).size()
)
display(pd.read_csv(out / "fit_failures.csv"))
display(results["scores"])

## Scale-independent summaries and model selection

MAE, RMSE and bias retain source units; they are not pooled across materials. Macro MASE and relative MAE improvements are unitless. Confirmation rankings are descriptive and do not replace the pre-frozen selection rule. At least 5% selection MAE improvement, 60% origin wins, six selection folds and positive gains in both 2021 and 2022 are required to displace LOCF.

In [ ]:
display(results["aggregate"])
display(results["selection"])
display(pd.read_csv(out / "frozen_policy_performance.csv"))
display(pd.read_csv(out / "regime_performance.csv"))

## Performance, rolling errors and tree importance

Tree importances describe final fitted challengers and are neither causal nor holdout feature selection. Correlated lags and different tree importance definitions require caution.

In [ ]:
for path in sorted(out.glob("*.png")):
    display(Image(filename=str(path)))

## Final point forecasts and recommendations

September 30, 2026 decision; December 2026, March 2027 and June 2027 target months. Models forecast across publication delays from June RM observations. These are assumption-based point forecasts, with no fabricated actual values or intervals.

In [ ]:
display(results["final"].loc[results["final"].selected])
print((out / "phase4_report.md").read_text())
print("MLflow parent run:", (out / "mlflow_run_id.txt").read_text())